# City population Markov Cahain
This is a simple implementation of a Markov Chain to model city populations over time. The model assumes that the population of a city can change based on certain probabilities.
A total of 20 cities are modeled, the population of each city can increase or decrease based on defined probabilities over iterations.
```python

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import sys
import os

dataset_path = kagglehub.dataset_download("justinboon/municipalities-of-the-netherlands")
print(f"Dataset downloaded to: {dataset_path}")

Dataset downloaded to: /Users/jadenvanrijswijk/.cache/kagglehub/datasets/justinboon/municipalities-of-the-netherlands/versions/7


In [24]:
df = pd.read_csv(os.path.join(dataset_path, 'municipalities_v7.csv'))
df = df[['municipality', 'province', 'population', 'surface_km2', 'latitude', 'longitude']]
RANDOM_SEED = 42
CITY_SAMPLE_SIZE = 20

print(df.size)
print(df.info())

41040
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6840 entries, 0 to 6839
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   municipality  6840 non-null   object 
 1   province      6840 non-null   object 
 2   population    376 non-null    float64
 3   surface_km2   6804 non-null   float64
 4   latitude      6840 non-null   float64
 5   longitude     6840 non-null   float64
dtypes: float64(4), object(2)
memory usage: 320.8+ KB
None


In [36]:
df = df.dropna()
df = df[df['province'] == 'Noord-Holland']
df = df[df['population'] > 0]
df = df.rename(columns={'municipality': 'city'}) 

# get top 20 highest population cities
df = df.nlargest(CITY_SAMPLE_SIZE, 'population').copy().reset_index(drop=True)
starting_city = df[df['city'] == 'Amstelveen'].iloc[0]

df

,city,province,population,surface_km2,distance_from_start_km
0,Amsterdam,Noord-Holland,853312.0,219.3,6.756099
1,Haarlem,Noord-Holland,155205.0,32.1,17.394498
2,Zaanstad,Noord-Holland,150911.0,83.2,18.188035
3,Haarlemmermeer,Noord-Holland,144166.0,185.3,13.364005
4,Alkmaar,Noord-Holland,94906.0,31.2,36.580076
5,Hilversum,Noord-Holland,86574.0,46.4,22.170569
6,Amstelveen,Noord-Holland,85135.0,44.1,0.000000
7,Purmerend,Noord-Holland,79552.0,24.6,23.450912
8,Hoorn,Noord-Holland,71741.0,53.3,38.987057
9,Velsen,Noord-Holland,67231.0,63.1,22.556555


In [ ]:
def latitude_longtitude_to_distance_km(lat1, lon1, lat2, lon2):
    """ Haversine formula to calculate distance between two lat/lon points in km.
        This function was made with AI assistance.
    """
    R = 6371  # Radius of the Earth in km
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c
    return distance

def log_scale_normalize(arr, x):
    """ Log scale normalize an array value to 0-1 range.
        This is used to reduce the impact of large outliers.
    """
    top_log = np.log(x + 1)
    bottom_log = np.log(np.max(arr) + 1)
    return 1 - (top_log / bottom_log)

# Compute distance matrix
distance_matrix = np.zeros((len(df), len(df)))
for i in range(len(df)):
    for j in range(len(df)):
        distance_matrix[i, j] = latitude_longtitude_to_distance_km(
            df.loc[i, 'latitude'], df.loc[i, 'longitude'],
            df.loc[j, 'latitude'], df.loc[j, 'longitude']
        )

In [ ]:
class MarkovChainNode:
    def __init__(self, city):
        self.name = city['city']
        self.population = city['population']
        self.city_index = df.index[df['city'] == self.name][0]
        self.city_distances = distance_matrix[self.city_index]
    
        self.max_connections = 5
        self.connection = []
        
        self.transition_weights = {
            'distance': 0.5,
            'population': 0.4,
            'surface_km2': 0.1,
        }
            
        
    def evaluate_attractiveness(self, other_city):
        """ Evaluate the attractiveness of another city based on distance, surface area and population.
            A high population and surface area increases attractiveness, while a high distance decreases it.
            The attractiveness is a value between 0 and 1.
        """
        distance_factor = log_scale_normalize(self.city_distances, self.city_distances[df.index[df['city'] == other_city.name][0]])
        population_factor = other_city.population / self.population
        surface_area_factor = other_city.surface_km2 / self.surface_km2        
        
        attractiveness = (
            self.transition_weights['population'] * population_factor +
            self.transition_weights['surface_km2'] * surface_area_factor +
            self.transition_weights['distance'] * distance_factor
        )
        
        return attractiveness

    def evolve_connections():
        """
        """
        pass
    
    def evolve_population(self):
        # Population change based on simple probabilities
        pass